# Transformer Sentence Embeddings — Kaggle Training

## Pipeline (run cells top-to-bottom)

| Phase | Cell | What it does | Output |
|-------|------|--------------|--------|
| 1 | 4 | MNR training on AllNLI pairs | `best_model.pt`, `vocab.pkl` |
| 2 | 4 | Evaluate best model on sts-222 val + test | `eval_results.json` |
| 3 | 4 | Triplet fine-tuning on AllNLI | `final_triplet_model.pt` |
| Display | 5 | Print evaluation metrics | — |
| Demo | 6 | Inference similarity demo | — |

## Before running
1. **Upload project** as a Kaggle dataset (containing `train.py`, `data.py`, `models/`, `losses/`, `evaluate_search.py`) and set `PROJECT_DIR` in Cell 2.
2. **Attach sts-222 dataset** (e.g. `amreisa9/sts-222`) and set `STS222_DIR` in Cell 3.
3. **AllNLI** is downloaded automatically from HuggingFace — no manual upload needed.
4. **Enable GPU**: Settings → Accelerator → GPU T4
5. **Enable Internet**: Settings → Internet → On (required for AllNLI download)

In [3]:
import shutil
import os

folder = "/kaggle/working/Transformer - Copy"
zip_file = "/kaggle/working/Transformer - Copy.zip"

if os.path.exists(folder):
    shutil.rmtree(folder)
    print("Deleted folder")

if os.path.exists(zip_file):
    os.remove(zip_file)
    print("Deleted zip file")

Deleted folder
Deleted zip file


In [4]:
!gdown "https://drive.google.com/uc?id=1pdcjYHSi95NQpAIWF2ar5JuKllxk7qGy"
!unzip "Transformer - Copy.zip"
# https://drive.google.com/file/d/1pdcjYHSi95NQpAIWF2ar5JuKllxk7qGy/view?usp=sharing

Downloading...
From: https://drive.google.com/uc?id=1pdcjYHSi95NQpAIWF2ar5JuKllxk7qGy
To: /kaggle/working/Transformer - Copy.zip
100%|██████████████████████████████████████| 94.4k/94.4k [00:00<00:00, 93.2MB/s]
Archive:  Transformer - Copy.zip
   creating: Transformer - Copy/
   creating: Transformer - Copy/checkpoints/
  inflating: Transformer - Copy/data.py  
  inflating: Transformer - Copy/evaluate_model.ipynb  
  inflating: Transformer - Copy/evaluate_search.py  
  inflating: Transformer - Copy/kaggle_running.ipynb  
  inflating: Transformer - Copy/local_running.ipynb  
   creating: Transformer - Copy/losses/
  inflating: Transformer - Copy/losses/contrastive_loss.py  
  inflating: Transformer - Copy/losses/cosent_loss.py  
  inflating: Transformer - Copy/losses/cosine_mse_loss.py  
  inflating: Transformer - Copy/losses/mnr_loss.py  
  inflating: Transformer - Copy/losses/triplet_loss.py  
  inflating: Transformer - Copy/losses/__init__.py  
   creating: Transformer - Copy/losses/_

In [1]:
!pip install -q datasets scipy tqdm

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


In [6]:
import os, sys, shutil

# ── Set to your Kaggle dataset slug containing the project files ──────────────
PROJECT_DIR = '/kaggle/input/datasets/amreisa9/transformer2'   # <-- update this
# ────────────────────────────────────────────────────────────────────────────

WORKING_DIR = '/kaggle/working/Transformer - Copy'

if os.path.isdir(PROJECT_DIR):
    print(f'Copying project files from {PROJECT_DIR} ...')
    for item in os.listdir(PROJECT_DIR):
        src = os.path.join(PROJECT_DIR, item)
        dst = os.path.join(WORKING_DIR, item)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    print('Done.')
else:
    print(f'[WARNING] PROJECT_DIR not found: {PROJECT_DIR}')
    print('Files must already be present in the working directory.')

sys.path.insert(0, WORKING_DIR)
print(f'sys.path[0] = {WORKING_DIR}')
print('Files:', [f for f in os.listdir(WORKING_DIR) if not f.startswith('.')][:20])

Copying project files from /kaggle/input/datasets/amreisa9/transformer2 ...
Done.
sys.path[0] = /kaggle/working/Transformer - Copy
Files: ['train_triplet.py', 'evaluate_search.py', 'train_mnr.py', 'losses', 'data', 'local_running.ipynb', 'checkpoints', 'train_compare.py', 'evaluate_model.ipynb', 'train_cosent.py', 'requirements.txt', 'train_cosine_mse.py', '__pycache__', 'models', 'kaggle_running.ipynb', 'train_contrastive.py', 'data.py', 'train.py', 'train_triplet_hard.py']


In [7]:
import os, shutil, pandas as pd

os.makedirs('data/AllNLI', exist_ok=True)
os.makedirs('data/sts-222', exist_ok=True)

# ── AllNLI ────────────────────────────────────────────────────────────────────
ALLNLI_KAGGLE = '/kaggle/input/datasets/amreisa9/transformer2/data/AllNLI/AllNLI.csv'
ALLNLI_PATH   = 'data/AllNLI/AllNLI.csv'

if os.path.exists(ALLNLI_PATH):
    n_rows = len(pd.read_csv(ALLNLI_PATH))
    print(f'AllNLI already present ({n_rows:,} rows).')

elif os.path.exists(ALLNLI_KAGGLE):
    shutil.copy(ALLNLI_KAGGLE, ALLNLI_PATH)
    n_rows = len(pd.read_csv(ALLNLI_PATH))
    print(f'Copied AllNLI from Kaggle dataset ({n_rows:,} rows).')

else:
    raise FileNotFoundError(
        f'AllNLI file not found:\n{ALLNLI_KAGGLE}\n'
        'Please attach the dataset to Kaggle.'
    )
# ── Hard Negatives ───────────────────────────────────────────────────────────
HARD_NEG_KAGGLE = '/kaggle/input/datasets/amreisa9/transformer2/data/AllNLI/train_hard_negatives.csv'
HARD_NEG_PATH   = 'data/AllNLI/train_hard_negatives.csv'

if os.path.exists(HARD_NEG_PATH):
    n_rows = len(pd.read_csv(HARD_NEG_PATH))
    print(f'Hard negatives already present ({n_rows:,} rows).')

elif os.path.exists(HARD_NEG_KAGGLE):
    shutil.copy(HARD_NEG_KAGGLE, HARD_NEG_PATH)
    n_rows = len(pd.read_csv(HARD_NEG_PATH))
    print(f'Copied hard negatives from Kaggle dataset ({n_rows:,} rows).')

else:
    raise FileNotFoundError(
        f'Hard negatives file not found:\n{HARD_NEG_KAGGLE}\n'
        'Please attach the dataset to Kaggle.'
    )

# ── STS-222 ───────────────────────────────────────────────────────────────────
STS222_DIR = '/kaggle/input/datasets/amreisa9/transformer2/data/sts-222'

for fname in ('stsb_validation.csv', 'stsb_test.csv'):
    dst = f'data/sts-222/{fname}'
    src = os.path.join(STS222_DIR, fname)

    if os.path.exists(dst):
        print(f'{dst} already present.')
        continue

    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied {fname}.')
    else:
        raise FileNotFoundError(f'Missing file: {src}')

print('\nData ready.')

AllNLI already present (961,328 rows).
Hard negatives already present (297,760 rows).
data/sts-222/stsb_validation.csv already present.
data/sts-222/stsb_test.csv already present.

Data ready.


In [12]:
# !rm /kaggle/working/checkpoints

rm: cannot remove '/kaggle/working/checkpoints': Is a directory


In [2]:
!python "/kaggle/working/Transformer - Copy/train_compare.py"

Device : cuda

Loading vocabulary from vocab.pkl ...
Vocab size : 146,470

  [MNR]  lr=3e-04  bs=128  epochs=20
  Params : 66,891,264  |  Training samples : 961,328
  Resumed: epoch=11  step=82621  best_rho=0.7243
  Epoch 12/20  lr=1.04e-04  loss=0.3996  val_rho=0.7272 *                       
  Epoch 13/20  lr=8.20e-05  loss=0.3615  val_rho=0.7266                         
  Epoch 14/20  lr=6.19e-05  loss=0.3289  val_rho=0.7302 *                       
  Epoch 15/20  lr=4.40e-05  loss=0.3004  val_rho=0.7297                         
  Epoch 16/20  lr=2.87e-05  loss=0.2774  val_rho=0.7310 *                       
  mnr 17/20:  23%|██████                    | 1743/7511 [07:51<23:59,  4.01it/s]^C
Traceback (most recent call last):                                              
  File "/kaggle/working/Transformer - Copy/train_compare.py", line 352, in <module>
    main()
  File "/kaggle/working/Transformer - Copy/train_compare.py", line 325, in main
    all_results[name] = run_approach(name,

In [26]:
# Runs all 3 phases in sequence:
#   Phase 1 — MNR training on AllNLI pairs         -> best_model.pt, vocab.pkl
#   Phase 2 — Evaluate best model on sts-222        -> eval_results.json
#   Phase 3 — Triplet fine-tuning on AllNLI         -> final_triplet_model.pt
!python "/kaggle/working/Transformer - Copy/train.py"

Device : cuda

Building vocabulary from AllNLI ...
Vocab size : 58,633
Parameters : 17,118,464


Phase 1 -- MNR Training on AllNLI
Training pairs : 277,230
  Epoch 01/20  lr=2.98e-04  loss=1.4680  val_rho=0.5719                         
  * best_model.pt saved  (val rho = 0.5719)
  Epoch 02/20  lr=2.93e-04  loss=1.0712  val_rho=0.6228                         
  * best_model.pt saved  (val rho = 0.6228)
  Epoch 03/20  lr=2.84e-04  loss=0.9422  val_rho=0.6371                         
  * best_model.pt saved  (val rho = 0.6371)
  Epoch 04/20  lr=2.72e-04  loss=0.8702  val_rho=0.6548                         
  * best_model.pt saved  (val rho = 0.6548)
  Epoch 05/20  lr=2.56e-04  loss=0.8186  val_rho=0.6537                         
  Epoch 06/20  lr=2.38e-04  loss=0.7806  val_rho=0.6725                         
  * best_model.pt saved  (val rho = 0.6725)
  Epoch 07/20  lr=2.18e-04  loss=0.7476  val_rho=0.6709                         
  Epoch 08/20  lr=1.97e-04  loss=0.7190  val_rho=0.6760  

In [46]:
#   Phase 3 — Triplet fine-tuning on          -> final_triplet_model.pt
!python "/kaggle/working/Transformer - Copy/train_triplet_hard.py"

Device : cuda

Loading vocabulary from /kaggle/working/Transformer - Copy/vocab.pkl ...
Vocab size : 58,633

Loading hard-negative triplets from data/AllNLI/train_hard_negatives.csv ...
Triplets : 297,760  |  Steps/epoch : 9,305

Loading pretrained weights from /kaggle/working/Transformer - Copy/best_model.pt ...
Pretrained weights loaded.
Baseline val Spearman = 0.6905

Triplet Fine-Tuning  |  epochs=3  margin=0.4
lr=5e-06  warmup=2791/27915 steps
Epoch 01/3: 100%|█| 9305/9305 [06:54<00:00, 22.44it/s, loss=0.1935, lr=4.22e-06]
  Epoch 01  loss=0.2345  val_rho=0.6218  lr=4.22e-06
Epoch 02/3: 100%|█| 9305/9305 [06:56<00:00, 22.35it/s, loss=0.1650, lr=1.51e-06]
  Epoch 02  loss=0.1803  val_rho=0.6127  lr=1.51e-06
Epoch 03/3: 100%|█| 9305/9305 [06:56<00:00, 22.32it/s, loss=0.1443, lr=0.00e+00]
  Epoch 03  loss=0.1708  val_rho=0.6109  lr=0.00e+00

Fine-tuning done.  Best val Spearman = 0.6905  (+0.0000 vs baseline)

Loading best checkpoint for final evaluation ...

  VAL -- data/sts-222/st

In [ ]:
# ── Show evaluation results (Phase 2 output) ──────────────────────────────────
import json

with open('eval_results.json') as f:
    results = json.load(f)

print('Evaluation results on sts-222 (best MNR model, before Triplet fine-tuning)\n')
for split, m in results.items():
    print(f'{split.upper()}')
    print(f"  Recall@1  : {m['recall@1']:.4f}")
    print(f"  Recall@5  : {m['recall@5']:.4f}")
    print(f"  Recall@10 : {m['recall@10']:.4f}")
    print(f"  MRR       : {m['mrr']:.4f}")
    print(f"  Spearman  : {m['spearman']:.4f}")
    print()

In [8]:
import os
import sys
import json
import pickle
import torch

PROJECT_DIR = "/kaggle/working/Transformer - Copy"
if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

from models.model.transformer import Transformer
from evaluate_search import evaluate, encode_sentences

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_CONFIG = {
    "max_len": 128,
    "d_model": 384,
    "n_layers": 6,
    "n_heads": 6,
    "d_ff": 1536,
    "pooling": "mean",
}

VAL_PATH = "/kaggle/working/Transformer - Copy/data/sts-222/stsb_validation.csv"
TEST_PATH = "/kaggle/working/Transformer - Copy/data/sts-222/stsb_test.csv"

BEST_PATH = "/kaggle/working/compare_runs/mnr/best_model.pt"

# ------------------------------------------------------------------
# Load Vocabulary
# ------------------------------------------------------------------

with open("/kaggle/working/vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

print(f"Vocab: {len(vocab):,} tokens")

# ------------------------------------------------------------------
# Build Model
# ------------------------------------------------------------------

model = Transformer(
    vocab_size=len(vocab),
    d_model=MODEL_CONFIG["d_model"],
    n_layers=MODEL_CONFIG["n_layers"],
    n_heads=MODEL_CONFIG["n_heads"],
    d_ff=MODEL_CONFIG["d_ff"],
    max_len=MODEL_CONFIG["max_len"],
    dropout=0.0,
    pooling=MODEL_CONFIG["pooling"],
).to(DEVICE)

checkpoint = torch.load(BEST_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint)

model.eval()

print("Loaded best_model.pt")

# ------------------------------------------------------------------
# Evaluation
# ------------------------------------------------------------------

results = {}

for split, path in [
    ("val", VAL_PATH),
    ("test", TEST_PATH),
]:
    if not os.path.exists(path):
        print(f"[SKIP] {path}")
        continue

    metrics = evaluate(
        model=model,
        csv_path=path,
        vocab=vocab,
        max_len=MODEL_CONFIG["max_len"],
        device=DEVICE,
        pos_threshold=0.65,
        batch_size=64,
    )

    results[split] = metrics

    print(
        f"\n{split.upper()}:\n"
        f"Recall@1  = {metrics['recall@1']:.4f}\n"
        f"Recall@5  = {metrics['recall@5']:.4f}\n"
        f"Recall@10 = {metrics['recall@10']:.4f}\n"
        f"MRR       = {metrics['mrr']:.4f}\n"
        f"Spearman  = {metrics['spearman']:.4f}"
    )

with open("eval_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nSaved -> eval_results.json")

# ------------------------------------------------------------------
# Inference Utilities
# ------------------------------------------------------------------

@torch.no_grad()
def embed(text: str) -> torch.Tensor:
    """
    Returns a single normalized embedding vector.
    Uses the exact same pipeline as evaluation.
    """
    emb = encode_sentences(
        model=model,
        sentences=[text],
        vocab=vocab,
        max_len=MODEL_CONFIG["max_len"],
        batch_size=1,
        device=DEVICE,
    )

    return emb[0].to(DEVICE)


@torch.no_grad()
def similarity(text1: str, text2: str) -> float:
    """
    Cosine similarity because embeddings are normalized.
    """
    emb1 = embed(text1)
    emb2 = embed(text2)

    return torch.dot(emb1, emb2).item()


# ------------------------------------------------------------------
# Demo
# ------------------------------------------------------------------

test_pairs = [
    ("A dog is running in the park", "A dog is playing outside"),
    ("The man is eating a sandwich", "A person is having lunch"),
    ("A woman is singing a song", "A lady is performing music"),
    ("The cat sat on the mat", "A dog is running in the park"),
    ("A man is riding a bicycle", "Someone is driving a car"),
    ("I love pizza", "The stock market crashed today"),
]

print(f'\n{"Sentence 1":<42} {"Sentence 2":<42} {"Sim":>8}')
print("-" * 100)

for s1, s2 in test_pairs:
    sim = similarity(s1, s2)

    bar_length = max(0, int((sim + 1) * 10))
    bar = "#" * bar_length

    print(
        f"{s1:<42} "
        f"{s2:<42} "
        f"{sim:>8.4f}  "
        f"{bar}"
    )

# # ------------------------------------------------------------------
# # Interactive Testing
# # ------------------------------------------------------------------

# while True:
#     print("\nEnter two sentences (or 'quit')")

#     s1 = input("Sentence 1: ").strip()

#     if s1.lower() in {"quit", "exit"}:
#         break

#     s2 = input("Sentence 2: ").strip()

#     score = similarity(s1, s2)

#     print(f"\nCosine Similarity = {score:.4f}")

Vocab: 146,470 tokens
Loaded best_model.pt
    Encoding 181018 sentences ...

VAL:
Recall@1  = 0.8105
Recall@5  = 0.9257
Recall@10 = 0.9430
MRR       = 0.8625
Spearman  = 0.7310
    Encoding 180953 sentences ...

TEST:
Recall@1  = 0.8057
Recall@5  = 0.9071
Recall@10 = 0.9324
MRR       = 0.8529
Spearman  = 0.7337

Saved -> eval_results.json

Sentence 1                                 Sentence 2                                      Sim
----------------------------------------------------------------------------------------------------
A dog is running in the park               A dog is playing outside                     0.5954  ###############
The man is eating a sandwich               A person is having lunch                     0.5970  ###############
A woman is singing a song                  A lady is performing music                   0.7454  #################
The cat sat on the mat                     A dog is running in the park                 0.1485  ###########
A man is riding